# CP2 CV/NLP Operator Notebook — Kelompok 7

Gunakan environment CP2 pada JupyterHub. Notebook mencari root repository dari lokasi notebook, kemudian menyediakan preflight, pengujian, audit data dan kontrol smoke/pilot. Simpan data, checkpoint dan prediksi per-ID pada folder privat. Status hasil dan keputusan tersedia di `docs/cp2/mulai-cp2.md`.

In [ ]:
from pathlib import Path
import os, subprocess, sys
repo = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'src/cp2/nlp_runner.py').is_file()), None)
assert repo is not None, 'Buka notebook dari checkout CP2'
os.chdir(repo)
print('Repository', repo)
print('Python', sys.version.split()[0])


## 1. Preflight
Pengecekan read-only. Bandingkan VRAM/RAM/disk dengan kebutuhan GPU sebelum menjalankan pelatihan.

In [ ]:
subprocess.run(['bash','scripts/cp2_ai_center_run.sh','preflight'],check=True)

## 2. Pengujian CP2
Sel ini menjalankan seluruh tes CP2 dan verifier artefak CP1. Smoke dengan data asli dicatat pada panduan CP2.

In [ ]:
subprocess.run(['bash','scripts/cp2_ai_center_run.sh','test'],check=True)

## 3. Inisialisasi & audit dataset
Membutuhkan akses ke sumber dataset pinned (atau cache metadata dan manifest CP1 yang sudah disalin). Hash/ukuran harus cocok. *Jangan* mengubah split.

In [ ]:
RUN_DATA_AUDIT = False  # ubah True hanya setelah izin akses dan workspace privat disiapkan
if RUN_DATA_AUDIT:
    for task in ['init-data','audit']:
        subprocess.run(['bash','scripts/cp2_ai_center_run.sh',task],check=True)
else:
    print('SKIPPED: data init/audit')

## 4. Kalibrasi komponen
Hanya jalankan ketika GPU bebas dan alokasi diizinkan. MHA sintetis bukan kalibrasi multimodal penuh. Simpan hasil di folder private.

In [ ]:
RUN_GPU_CALIBRATION = False
if RUN_GPU_CALIBRATION:
    for task in ['calibrate-attention','calibrate-b0','calibrate-vision']:
        subprocess.run(['bash','scripts/cp2_ai_center_run.sh',task],check=True)
else:
    print('SKIPPED: GPU calibration')

## 5. Smoke/Pilot B0
Jalankan setelah audit data dan pemeriksaan kapasitas selesai. Hasil smoke/pilot dicatat dengan scope masing-masing.

In [ ]:
RUN_SMOKE = False
RUN_PILOT = False
if RUN_SMOKE:
    subprocess.run(['bash','scripts/cp2_ai_center_run.sh','smoke-b0'],check=True)
if RUN_PILOT:
    subprocess.run(['bash','scripts/cp2_ai_center_run.sh','pilot-b0'],check=True)
if not (RUN_SMOKE or RUN_PILOT): print('SKIPPED: training')

## 6. Training penuh
Full training memakai protokol yang dikunci dan dicatat dalam config run. Gunakan terminal sesuai panduan `docs/cp2/mulai-cp2.md` agar pemantauan berlangsung sepanjang run.

## 7. Smoke/Pilot multimodal B1–M3
Pilih satu varian dan aksi. Nilai `RUN_MULTIMODAL` mengaktifkan aksi yang dipilih. Skrip memeriksa kapasitas GPU dan folder privat sebelum menjalankan model.

In [ ]:
RUN_MULTIMODAL = False
MULTIMODAL_VARIANT = 'M3'
MULTIMODAL_ACTION = 'smoke'
assert MULTIMODAL_VARIANT in ('B1', 'B2', 'M1', 'M2', 'M3')
assert MULTIMODAL_ACTION in ('smoke', 'pilot')
if RUN_MULTIMODAL:
    action = f'{MULTIMODAL_ACTION}-{MULTIMODAL_VARIANT.lower()}'
    subprocess.run(['bash', 'scripts/cp2_mm_run.sh', action], check=True)
else:
    print('Pilih varian dan aksi, lalu aktifkan RUN_MULTIMODAL saat siap')
